In [18]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "runs" / "experiments.sqlite3").exists())
sys.path.insert(0, str(ROOT))
from agentsq.analysis import db

conn = db.connect()
print(db.sync(conn)["total"], "runs in", db.DB_PATH)
conn.close()

conn = sqlite3.connect(db.DB_PATH)

134 runs in /home/jsh27/agent-monitor/agent-squared/runs/experiments.sqlite3


In [19]:
results = pd.read_sql_query("""
    SELECT run, run_group, batch, condition, agent_model AS model,
           question_id, question_title, correct, agent_answer AS answer
      FROM runs
     WHERE experiment = 'benchmark_runs'
  ORDER BY model, batch, condition, question_title
""", conn)

results["run_folder"] = results["run"].str.split("~").str[0]
results["correct"] = results["correct"].astype("boolean")  

results[["model", "batch", "condition", "question_title", "correct", "run_folder"]]

,model,batch,condition,question_title,correct,run_folder
0,inkling-bf16,1,baseline,diving_world_cup,True,batch1-0-baseline-inkling-bf16-inkling-bf16-me...
1,inkling-bf16,1,baseline,philosophy_faculty,True,batch1-0-baseline-inkling-bf16-inkling-bf16-me...
2,inkling-bf16,1,baseline,polymer_startup,False,batch1-0-baseline-inkling-bf16-inkling-bf16-me...
3,inkling-bf16,1,baseline,rule_of_law_debate,True,batch1-0-baseline-inkling-bf16-inkling-bf16-me...
4,inkling-bf16,1,baseline,shipping_ceo,False,batch1-0-baseline-inkling-bf16-inkling-bf16-me...
...,...,...,...,...,...,...
95,nemotron-3-ultra,2,broken,automotive_host_city,False,batch1-0-broken-nemotron-3-ultra-nemotron-3-ul...
96,nemotron-3-ultra,2,broken,lifetime_achievement_award,False,batch1-0-broken-nemotron-3-ultra-nemotron-3-ul...
97,nemotron-3-ultra,2,broken,nsf_3d_vision_grant,False,batch1-0-broken-nemotron-3-ultra-nemotron-3-ul...
98,nemotron-3-ultra,2,broken,planetary_defense_smallsat,True,batch1-0-broken-nemotron-3-ultra-nemotron-3-ul...


In [20]:
graded = results.dropna(subset=["correct"]).assign(correct=lambda d: d["correct"].astype(int))

summary = (graded
    .groupby(["model", "batch", "condition"])["correct"]
    .agg(n="count", n_correct="sum", mean="mean", std="std")
    .round(3))
summary

n  n_correct  mean    std
model            batch condition                            
inkling-bf16     1     baseline    5          3  0.60  0.548
                       broken      5          1  0.20  0.447
                 2     baseline    5          3  0.60  0.548
                       broken      5          0  0.00  0.000
                 3     baseline   10          5  0.50  0.527
                       broken     10          3  0.30  0.483
                 4     baseline   20          5  0.25  0.444
                       broken     20          3  0.15  0.366
nemotron-3-ultra 1     baseline    5          1  0.20  0.447
                       broken      5          0  0.00  0.000
                 2     baseline    5          4  0.80  0.447
                       broken      5          1  0.20  0.447

In [21]:
wide = summary.apply(lambda r: f"{r['mean']:.2f} ± {r['std']:.2f}  (n={int(r['n'])})", axis=1)
wide.unstack(["batch", "condition"]).sort_index(axis=1).fillna("–")   # – = no run yet

batch                              1                                       2  \
condition                   baseline              broken            baseline   
model                                                                          
inkling-bf16      0.60 ± 0.55  (n=5)  0.20 ± 0.45  (n=5)  0.60 ± 0.55  (n=5)   
nemotron-3-ultra  0.20 ± 0.45  (n=5)  0.00 ± 0.00  (n=5)  0.80 ± 0.45  (n=5)   

batch                                                   3  \
condition                     broken             baseline   
model                                                       
inkling-bf16      0.00 ± 0.00  (n=5)  0.50 ± 0.53  (n=10)   
nemotron-3-ultra  0.20 ± 0.45  (n=5)                    –   

batch                                                    4  \
condition                      broken             baseline   
model                                                        
inkling-bf16      0.30 ± 0.48  (n=10)  0.25 ± 0.44  (n=20)   
nemotron-3-ultra                    –                    –   

batch                                  
condition                      broken  
model                                  
inkling-bf16      0.15 ± 0.37  (n=20)  
nemotron-3-ultra                    –

In [22]:
graded = results.dropna(subset=["correct"]).assign(correct=lambda d: d["correct"].astype(int))

summary = (graded
    .groupby(["condition"])["correct"]
    .agg(n="count", n_correct="sum", mean="mean", std="std")
    .round(3))
summary

,n,n_correct,mean,std
condition,,,,
baseline,50,21,0.42,0.499
broken,50,8,0.16,0.370
